# Synthetic dataset training run

# Continual Learning — Colab Training Notebook

**Before running:**
1. Set runtime to **GPU** (Runtime → Change runtime type → T4 GPU or better)
2. Edit the **Configuration** cell below
3. Run all cells top-to-bottom

Checkpoints and the dataset cache are stored on **Google Drive** and survive session restarts.
If a checkpoint already exists for your `RUN_ID`, training will automatically resume from it.

## Configuration
Edit the values below before running the notebook.

In [62]:
# ============================================================
#  CONFIGURE EVERYTHING HERE
# ============================================================

# Training method — choose one:
#   "full_ft"  — fine-tune all parameters (catastrophic forgetting baseline)
#   "lora"     — parameter-efficient LoRA adapters
#   "smf"      — frozen backbone + sparse trainable memory
#   "casm"     — versioned memory bank with contradiction-aware routing
METHOD = "smf"

# HuggingFace model (must have approved access)
MODEL_NAME = "meta-llama/Llama-3.2-1B"

# Unique experiment name — checkpoints are stored under this ID
RUN_ID = "llama1b_synthetic_smf_01"

# Periods to train — full sequence is all four in order
# Each period is one temporal snapshot of the synthetic dataset
PERIODS = ["2018", "2020"]  # full run: ["2018", "2020", "2022", "2024"]

# Google Drive folder — checkpoints and dataset cache go here
DRIVE_DIR = "/content/drive/MyDrive/continual_learning/synthetic"

# ---- Dataset source ----
# True  → training passages come from data/augmented/synthetic/<period>.csv
# False → training passages come from data/passages.json (thin templates, dev/testing only)
USE_AUGMENTED_DATA = True

# ---- Dataset fraction ----
# Fraction of the dataset to use for both training and evaluation.
# The fraction is applied proportionally and independently to the changed and
# unchanged probe splits, so training and evaluation always cover the same
# examples.  None means use all data; 0.1 means use 10% of each split.
DATASET_FRACTION = None  # synthetic dataset is small (605 facts) — use all data

# ---- Core hyperparameters ----
BATCH_SIZE              = 1
GRAD_ACCUM_STEPS        = 4       # effective batch size = 4
LEARNING_RATE           = 5e-4
EPOCHS_PER_PERIOD       = 100     # passes over all passages
MAX_PASSAGES_PER_PERIOD = None    # use all available passages
PRECISION               = "bfloat16"
SEED                    = 42

# ---- Activation capture ----
# Records per-layer hidden-state activations after each period.
# Output saved to periods/<period>/activations.pt and activation_metadata.json
# alongside eval outputs — copy to another machine for plotting, no GPU needed.
CAPTURE_ACTIVATIONS = True

# ---- LoRA settings (only used when METHOD == "lora") ----
LORA_R              = 16
LORA_ALPHA          = 32
LORA_DROPOUT        = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]

# ---- SMF settings (only used when METHOD == "smf") ----
# Llama-3.2-1B has 16 transformer layers (indices 0-15)
SMF_MEMORY_SIZE            = 64
SMF_SPARSITY_RATIO         = 0.1
SMF_UPDATE_LAYERS          = [4, 8, 12]  # mid-to-late layers
SMF_REGULARIZATION_WEIGHT  = 0.01
SMF_LEARNING_RATE          = 1e-3

# ---- CASM settings (only used when METHOD == "casm") ----
CASM_NUM_SLOTS              = 8
CASM_ROUTER_HIDDEN_SIZE     = 256
CASM_TOP_K                  = 2
CASM_ROUTER_TEMPERATURE     = 1.0
CASM_SPARSITY_WEIGHT        = 0.01
CASM_OVERLAP_WEIGHT         = 0.01
CASM_BRANCH_ON_CONTRADICTION = True
CASM_MEMORY_SIZE            = 64
CASM_ROUTER_TYPE            = 'mlp'  # 'mlp' = CASMRouter (learned, default), 'similarity' = SimilarityRouter (cosine, no training)

## Setup

In [63]:
# Mount Google Drive for persistent checkpoint and dataset storage
from google.colab import drive
drive.mount("/content/drive")

import os

CHECKPOINT_DIR   = os.path.join(DRIVE_DIR, "checkpoints")
DATASET_CACHE_DIR = os.path.join(DRIVE_DIR, "dataset_cache")
REPO_DIR         = "/content/Continual-Learning"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(DATASET_CACHE_DIR, exist_ok=True)

print(f"Checkpoint dir:  {CHECKPOINT_DIR}")
print(f"Dataset cache:   {DATASET_CACHE_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint dir:  /content/drive/MyDrive/continual_learning/checkpoints
Dataset cache:   /content/drive/MyDrive/continual_learning/dataset_cache


In [64]:
# Clone the repo (or pull if already cloned this session)
import subprocess

REPO_URL = "https://github.com/athirai-s/Continual-Learning"

if os.path.exists(os.path.join(REPO_DIR, ".git")):
    print("Repo already cloned — pulling latest changes...")
    result = subprocess.run(
        ["git", "-C", REPO_DIR, "pull"],
        capture_output=True, text=True
    )
    print(result.stdout.strip() or result.stderr.strip())
else:
    print(f"Cloning {REPO_URL} ...")
    result = subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        capture_output=True, text=True
    )
    print(result.stdout.strip() or result.stderr.strip())
    if result.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{result.stderr}")

print("Repo ready.")

Repo already cloned — pulling latest changes...
Already up to date.
Repo ready.


In [65]:
# Install dependencies not pre-installed on Colab
# transformers 5.x is required — Colab ships with 4.x by default
!pip install -q "transformers>=5.3.0" "peft>=0.18.1" "trl>=0.29.0" "datasets>=4.6.1"

# Add the repo to the Python path so imports work
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Dependencies installed. Repo on path.")

Dependencies installed. Repo on path.


## Dataset

Training passages come from `data/augmented/synthetic/<period>.csv` — these are
committed directly in the repo, so no download is needed.

Evaluation probes come from `data/probes.json`, which is also committed in the repo.
`SyntheticDataset` loads this file directly — no zip extraction needed.
The synthetic dataset contains 605 facts with ~150 changed probes per period boundary.

In [66]:
import os

if USE_AUGMENTED_DATA:
    # Verify augmented training CSVs (data/augmented/synthetic/<period>.csv)
    aug_dir = os.path.join(REPO_DIR, "data", "augmented", "synthetic")
    for period in PERIODS:
        csv_path = os.path.join(aug_dir, f"{period}.csv")
        assert os.path.exists(csv_path), f"Augmented CSV missing: {csv_path}"
        print(f"  {period}.csv: OK ({os.path.getsize(csv_path):,} bytes)")

    # Verify probes file
    probes_json = os.path.join(REPO_DIR, "data", "probes.json")
    assert os.path.exists(probes_json), (
        f"probes.json not found at {probes_json}. "
        "Run data/build_probes.py to generate it."
    )
    print(f"probes.json: OK ({os.path.getsize(probes_json):,} bytes)")
    print("
All dataset files present — no download needed.")
else:
    # Verify thin-template files for dev/testing
    probes_json   = os.path.join(REPO_DIR, "data", "probes.json")
    passages_json = os.path.join(REPO_DIR, "data", "passages.json")
    assert os.path.exists(probes_json),   f"probes.json not found at {probes_json}"
    assert os.path.exists(passages_json), f"passages.json not found at {passages_json}"
    print(f"probes.json:   OK ({os.path.getsize(probes_json):,} bytes)")
    print(f"passages.json: OK ({os.path.getsize(passages_json):,} bytes)")


TWiki_Probes.zip: OK (620,459 bytes)
  aug_sep.csv: OK (204,610 bytes)
  sep_oct.csv: OK (215,576 bytes)

All dataset files present — no download needed.


## HuggingFace Authentication

Required for gated models like Llama. Log in with a token that has **read** access to
`meta-llama/Llama-3.2-1B`. You can create a token at https://huggingface.co/settings/tokens

In [67]:
from huggingface_hub import login
login()  # will prompt for your HuggingFace access token

## Training

In [68]:
import os

if USE_AUGMENTED_DATA:
    aug_dir = os.path.join(REPO_DIR, "data", "augmented", "synthetic")
    all_ok = True
    for period in PERIODS:
        path = os.path.join(aug_dir, f"{period}.csv")
        exists = os.path.exists(path)
        size = os.path.getsize(path) if exists else 0
        status = "OK" if exists else "MISSING"
        print(f"  {period}.csv: {status} ({size:,} bytes)")
        if not exists:
            all_ok = False

    if all_ok:
        print("\nAll augmented CSVs found — training will use augmented passages.")
    else:
        raise FileNotFoundError(
            "One or more augmented CSVs are missing. "
            "Check that data/augmented/synthetic/<period>.csv files are committed to the repo."
        )
else:
    print("USE_AUGMENTED_DATA=False — training will use data/passages.json (thin templates).")

  aug_sep.csv: OK (204,610 bytes)
  sep_oct.csv: OK (215,576 bytes)

All augmented CSVs found — training will use augmented passages.


In [69]:
# Build TrainConfig from configuration variables set at the top
from training.train_config import TrainConfig

config_kwargs = dict(
    run_id=RUN_ID,
    model_name=MODEL_NAME,
    method=METHOD,
    dataset_name="synthetic",
    precision=PRECISION,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    epochs_per_period=EPOCHS_PER_PERIOD,
    max_passages_per_period=MAX_PASSAGES_PER_PERIOD,
    dataset_fraction=DATASET_FRACTION,
    log_every_n_steps=10,
    eval_after_each_period=True,
    capture_activations=CAPTURE_ACTIVATIONS,
    seed=SEED,
)

if METHOD == "lora":
    config_kwargs.update(
        lora_r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        lora_target_modules=LORA_TARGET_MODULES,
    )
elif METHOD == "smf":
    config_kwargs.update(
        smf_memory_size=SMF_MEMORY_SIZE,
        smf_sparsity_ratio=SMF_SPARSITY_RATIO,
        smf_update_layers=SMF_UPDATE_LAYERS,
        smf_regularization_weight=SMF_REGULARIZATION_WEIGHT,
        smf_learning_rate=SMF_LEARNING_RATE,
        smf_freeze_backbone=True,
    )
elif METHOD == "casm":
    config_kwargs.update(
        casm_num_slots=CASM_NUM_SLOTS,
        casm_router_hidden_size=CASM_ROUTER_HIDDEN_SIZE,
        casm_top_k=CASM_TOP_K,
        casm_router_temperature=CASM_ROUTER_TEMPERATURE,
        casm_sparsity_weight=CASM_SPARSITY_WEIGHT,
        casm_overlap_weight=CASM_OVERLAP_WEIGHT,
        casm_branch_on_contradiction=CASM_BRANCH_ON_CONTRADICTION,
        casm_memory_size=CASM_MEMORY_SIZE,
        casm_router_type=CASM_ROUTER_TYPE,
    )

cfg = TrainConfig(**config_kwargs)
cfg.validate()

print(f"Method:               {cfg.method}")
print(f"Model:                {cfg.model_name}")
print(f"Periods:              {PERIODS}")
print(f"Batch size:           {cfg.batch_size}")
print(f"Grad accum steps:     {cfg.grad_accum_steps}  (eff. batch = {cfg.batch_size * cfg.grad_accum_steps})")
print(f"Learning rate:        {cfg.learning_rate}")
print(f"Epochs per period:    {cfg.epochs_per_period}")
print(f"Max passages:         {cfg.max_passages_per_period}")
print(f"Dataset fraction:     {cfg.dataset_fraction if cfg.dataset_fraction is not None else 'all (None)'}")
print(f"Capture activations:  {cfg.capture_activations}")
print(f"Checkpoint dir:       {CHECKPOINT_DIR}")

Method:               smf
Model:                meta-llama/Llama-3.2-1B
Periods:              ['aug_sep', 'sep_oct']
Batch size:           1
Grad accum steps:     4  (eff. batch = 4)
Learning rate:        0.0005
Epochs per period:    100
Max passages:         None
Dataset fraction:     0.05
Capture activations:  True
Checkpoint dir:       /content/drive/MyDrive/continual_learning/checkpoints


In [70]:
# Detect existing checkpoint for automatic resume
import json

RESUME_FROM = None
run_root = os.path.join(CHECKPOINT_DIR, RUN_ID)
latest_json = os.path.join(run_root, "latest.json")

if os.path.exists(latest_json):
    with open(latest_json) as f:
        pointer = json.load(f)
    last_period = pointer.get("last_period", "unknown")
    print(f"Existing checkpoint found (last period: {last_period}).")
    print(f"Training will resume from: {run_root}")
    RESUME_FROM = run_root
else:
    print("No existing checkpoint — starting fresh.")

No existing checkpoint — starting fresh.


In [ ]:
# ── Data verification ─────────────────────────────────────────────────────
# Loads the first period's data without starting training so you can confirm
# the right passages and probes are in place before the run begins.
from casf_dataset_api import SyntheticDataset

print(f"Method: {cfg.method}")
print(f"Dataset: {cfg.dataset_name}")
print(f"Periods: {PERIODS}")
print()

for period in PERIODS:
    ds = SyntheticDataset(period)
    ds.load("changed")
    ds.load("unchanged")

    passages   = ds.get_train_passages()
    changed    = ds.get_probes("changed")
    unchanged  = ds.get_probes("unchanged")

    print(f"Period: {period}")
    print(f"  Training passages : {len(passages)}")
    print(f"  Changed probes    : {len(changed)}")
    print(f"  Unchanged probes  : {len(unchanged)}")

    if passages:
        print(f"  Sample passage    : {passages[0][:120]!r}")
    if changed:
        p = changed[0]
        print(f"  Sample changed    : [{p.subject}] {p.relation} -> {p.current_value!r}  (was {p.previous_value!r})")
    if unchanged:
        p = unchanged[0]
        print(f"  Sample unchanged  : [{p.subject}] {p.relation} -> {p.current_value!r}")
    print()

print("Data looks correct — proceed to training.")


In [71]:
from training.train_runner import (
    run_training,
    build_real_model_and_tokenizer,
    load_real_model_and_tokenizer,
    build_augmented_dataset,
    build_real_dataset,
)

dataset_factory = build_augmented_dataset if USE_AUGMENTED_DATA else build_real_dataset
print(f"Dataset factory: {'augmented CSVs (data/augmented/synthetic/)' if USE_AUGMENTED_DATA else 'data/passages.json (thin templates)'}")

results = run_training(
    cfg,
    model_factory=build_real_model_and_tokenizer,
    resume_model_factory=load_real_model_and_tokenizer,
    dataset_factory=dataset_factory,
    checkpoint_dir=CHECKPOINT_DIR,
    training_units=PERIODS,
    resume_from=RESUME_FROM,
)

print("\n" + "="*50)
print("TRAINING SUMMARY")
print("="*50)
for i, r in enumerate(results):
    period = PERIODS[i] if i < len(PERIODS) else f"period_{i}"
    print(f"\n{period}:")
    loss = r.get("train_loss_final")
    print(f"  Final loss:        {loss:.4f}" if loss is not None else "  Final loss:        N/A")
    print(f"  Passages trained:  {r.get('n_passages_trained', 'N/A')}")
    print(f"  Train time (s):    {r.get('train_duration_sec', 0):.1f}")
    if "evaluation" in r:
        for split, eval_result in r["evaluation"].items():
            if isinstance(eval_result, dict):
                plasticity = eval_result.get("plasticity")
                stability  = eval_result.get("stability")
                token_f1   = eval_result.get("token_f1")
            else:
                plasticity = eval_result.plasticity
                stability  = eval_result.stability
                token_f1   = eval_result.token_f1
            p = f"{plasticity:.3f}" if plasticity is not None else "N/A"
            s = f"{stability:.3f}"  if stability  is not None else "N/A"
            f = f"{token_f1:.3f}"   if token_f1   is not None else "N/A"
            print(f"  Eval [{split:9s}]: plasticity={p}  stability={s}  token_f1={f}")
    print(f"  Checkpoint:        {r.get('checkpoint_path', 'N/A')}")

Dataset factory: augmented CSVs
Training config:
TrainConfig(model_name='meta-llama/Llama-3.2-1B', method='smf', dataset_name='temporal_wiki', precision='bfloat16', learning_rate=0.0005, batch_size=1, grad_accum_steps=4, epochs_per_period=100, grad_clip=1.0, warmup_steps=100, min_passage_length=100, deduplicate=True, weighted_sampling=False, max_passages_per_period=None, dataset_fraction=0.05, shuffle_by_relation=True, contradiction_batch_frac=0.25, run_id='llama1b_augmented_dataset_smf_02', log_every_n_steps=10, eval_after_each_period=True, capture_activations=True, checkpoint_dir='/content/drive/MyDrive/continual_learning/checkpoints', checkpoint_every_n_optimizer_steps=None, seed=42, lora_r=None, lora_alpha=None, lora_dropout=0.05, lora_target_modules=None, smf_memory_size=64, smf_sparsity_ratio=0.1, smf_update_layers=[4, 8, 12], smf_regularization_weight=0.01, smf_freeze_backbone=True, smf_learning_rate=0.001, casm_num_slots=None, casm_router_hidden_size=None, casm_top_k=None, casm

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Saved config to /content/drive/MyDrive/continual_learning/checkpoints/llama1b_augmented_dataset_smf_02_config.json


=== Training unit: aug_sep ===
  dataset_fraction=0.050: 15/288 changed probes, 88/1741 unchanged probes, 103 training passages
  Passages:      4100 (100 epoch(s))
  Optimizer steps: 1025  |  eff_batch=4  |  lr=0.00e+00
  Device:          cuda
  step   10/1025 (  1.0%)  loss=3.7081  lr=1.00e-04    604 tok/s  elapsed=00:01  ETA=03:12
  step   20/1025 (  2.0%)  loss=3.6164  lr=2.00e-04    516 tok/s  elapsed=00:03  ETA=03:09
  step   30/1025 (  2.9%)  loss=2.9094  lr=3.00e-04    543 tok/s  elapsed=00:05  ETA=03:08
  step   40/1025 (  3.9%)  loss=2.6255  lr=4.00e-04    589 tok/s  elapsed=00:07  ETA=03:07
  step   50/1025 (  4.9%)  loss=2.4275  lr=5.00e-04    557 tok/s  elapsed=00:09  ETA=03:05
  step   60/1025 (  5.9%)  loss=1.7875  lr=6.00e-04    547 tok/s  elapsed=00:11  ETA=03:03
  step   70/1025 (  6.8%)  loss=1.8338  lr=7.00e-04    541 tok/s  elapsed=00:13  ETA=03:00
 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  dataset_fraction=0.050: 15/288 changed probes, 88/1741 unchanged probes, 103 training passages
Activations saved: 20 probes × 16 layers → /content/drive/MyDrive/continual_learning/checkpoints/llama1b_augmented_dataset_smf_02/periods/aug_sep/activations.pt
Training result:
  Final loss: 0.30002421885728836
  Passages trained: 4100
  Contradiction passages: 0
  Train duration (sec): 194.57
Checkpoint saved to: /content/drive/MyDrive/continual_learning/checkpoints/llama1b_augmented_dataset_smf_02/checkpoints/ckpt-000001

=== Training unit: sep_oct ===
  dataset_fraction=0.050: 15/294 changed probes, 92/1824 unchanged probes, 107 training passages
  Passages:      4200 (100 epoch(s))
  Optimizer steps: 1050  |  eff_batch=4  |  lr=0.00e+00
  Device:          cuda
  step   10/1050 (  1.0%)  loss=4.2569  lr=1.00e-04    581 tok/s  elapsed=00:02  ETA=03:28
  step   20/1050 (  1.9%)  loss=3.3320  lr=2.00e-04    659 tok/s  elapsed=00:03  ETA=03:21
  step   30/1050 (  2.9%)  loss=2.6993  lr=3.00

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  dataset_fraction=0.050: 15/294 changed probes, 92/1824 unchanged probes, 107 training passages
Activations saved: 20 probes × 16 layers → /content/drive/MyDrive/continual_learning/checkpoints/llama1b_augmented_dataset_smf_02/periods/sep_oct/activations.pt
Training result:
  Final loss: 0.2580132335424423
  Passages trained: 4200
  Contradiction passages: 0
  Train duration (sec): 201.24
Checkpoint saved to: /content/drive/MyDrive/continual_learning/checkpoints/llama1b_augmented_dataset_smf_02/checkpoints/ckpt-000002

Done.

TRAINING SUMMARY

aug_sep:
  Final loss:        0.3000
  Passages trained:  4100
  Train time (s):    194.6
  Eval [changed  ]: plasticity=0.000  stability=0.000  token_f1=0.000
  Eval [unchanged]: plasticity=0.000  stability=0.148  token_f1=0.003
  Checkpoint:        /content/drive/MyDrive/continual_learning/checkpoints/llama1b_augmented_dataset_smf_02/checkpoints/ckpt-000001

sep_oct:
  Final loss:        0.2580
  Passages trained:  4200
  Train time (s):    201

In [72]:
# Print eval summary from the completed run (no retraining needed)
print("\n" + "="*50)
print("EVAL SUMMARY")
print("="*50)
for i, r in enumerate(results):
    period = PERIODS[i] if i < len(PERIODS) else f"period_{i}"
    print(f"\n{period}:")
    if "evaluation" in r:
        for split, eval_result in r["evaluation"].items():
            if isinstance(eval_result, dict):
                plasticity = eval_result.get("plasticity")
                stability  = eval_result.get("stability")
                token_f1   = eval_result.get("token_f1")
            else:
                plasticity = eval_result.plasticity
                stability  = eval_result.stability
                token_f1   = eval_result.token_f1
            p = f"{plasticity:.3f}" if plasticity is not None else "N/A"
            s = f"{stability:.3f}"  if stability  is not None else "N/A"
            f = f"{token_f1:.3f}"   if token_f1   is not None else "N/A"
            print(f"  [{split:9s}]: plasticity={p}  stability={s}  token_f1={f}")
    else:
        print("  (no evaluation results)")


EVAL SUMMARY

aug_sep:
  [changed  ]: plasticity=0.000  stability=0.000  token_f1=0.000
  [unchanged]: plasticity=0.000  stability=0.148  token_f1=0.003

sep_oct:
  [changed  ]: plasticity=0.200  stability=0.000  token_f1=0.036
  [unchanged]: plasticity=0.000  stability=0.196  token_f1=0.016


In [80]:
import torch
from training.train_runner import load_real_model_and_tokenizer

checkpoint_path = results[0]["checkpoint_path"]
model, tokenizer = load_real_model_and_tokenizer(cfg, checkpoint_path)
model.eval()

device = next(model.parameters()).device

sample_dataset = dataset_factory(PERIODS[-1], cfg)
sample_dataset.load("changed")
probes = sample_dataset.get_probes("changed")[:5]

for probe in probes:
    inputs = tokenizer(probe.prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
        )
    generated = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    print(f"Prompt:   {probe.prompt}")
    print(f"Expected: {probe.ground_truth}")
    print(f"Got:      {generated!r}")
    print()

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


  dataset_fraction=0.050: 15/294 changed probes, 92/1824 unchanged probes, 107 training passages


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Prompt:   The publisher of Jacquelyn Reingold is
Expected: Ensemble Studio Theatre
Got:      'a member of the American'



...

Prompt:   A participant in Battle of Cnidus is
Expected: Conon
Got:      'a player who has a'



Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Prompt:   The occupation of Susannah M. Porter is
Expected: geobiologist
Got:      'a story of a woman'



Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Prompt:   The occupation of Jean Fleming is
Expected: biochemist
Got:      'a story of a young'

Prompt:   The composer of The Matrix Resurrections is
Expected: Tom Tykwer
Got:      'a man who has been'



In [74]:
from collections import defaultdict

def param_summary(model):
    total_params = 0
    trainable_params = 0
    groups = defaultdict(lambda: {"total": 0, "trainable": 0})

    for name, param in model.named_parameters():
        n = param.numel()
        top = name.split(".")[0]
        total_params += n
        groups[top]["total"] += n
        if param.requires_grad:
            trainable_params += n
            groups[top]["trainable"] += n

    col = 42
    print(f"\n{'Module':<{col}} {'Total params':>14} {'Trainable':>12} {'%':>7}")
    print("-" * (col + 36))
    for group, counts in sorted(groups.items()):
        t, tr = counts["total"], counts["trainable"]
        pct = 100 * tr / t if t else 0
        marker = "  <-- updated" if tr > 0 else ""
        print(f"  {group:<{col-2}} {t:>14,} {tr:>12,} {pct:>6.1f}%{marker}")
    print("-" * (col + 36))
    pct = 100 * trainable_params / total_params if total_params else 0
    print(f"  {'TOTAL':<{col-2}} {total_params:>14,} {trainable_params:>12,} {pct:>6.1f}%")
    print()
    print(f"Trainable: {trainable_params:,} / {total_params:,}  ({pct:.4f}% of model)")

param_summary(model)


Module                                       Total params    Trainable       %
------------------------------------------------------------------------------
  backbone                                  1,235,814,400            0    0.0%
  memory_blocks                                   786,624      786,624  100.0%  <-- updated
------------------------------------------------------------------------------
  TOTAL                                     1,236,601,024      786,624    0.1%

Trainable: 786,624 / 1,236,601,024  (0.0636% of model)
